# Engram P2.6: GPT-4o as the fact writer, same 200 sentences as the Gemma runs

GPT-4o (`gpt-4o-2024-08-06`) writes facts for the same 200 blind-labelled sentences, in the free line form and in the forced form. Same prompts, validator, scorer and pass rule as Gemma 3 12B; the rule is in `results/p26b/rule_gpt4o.md`, committed before this run. Benchmark data only.

**No GPU needed.** About 5 minutes, well under a dollar of OpenAI usage. Add your OpenAI key as a secret named `OPENAI_API_KEY` (Kaggle: Add-ons > Secrets, and Settings > Turn on internet; Colab: the key icon), or paste it when asked.

In [ ]:
COMMIT = "e87d52d"
import os, subprocess, sys
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
SAVE = f"{WORK}/engram_results"
os.makedirs(SAVE, exist_ok=True); os.chdir(WORK)
subprocess.run("rm -rf Engram; git clone -q https://github.com/manjunathhari/Engram.git", shell=True, check=True)
os.chdir(f"{WORK}/Engram")
subprocess.run(f"git checkout -q {COMMIT}", shell=True, check=True)
subprocess.run("pip install -q openai", shell=True, check=True)
key = os.environ.get("OPENAI_API_KEY", "")
for get in (lambda: __import__("google.colab.userdata", fromlist=["get"]).get("OPENAI_API_KEY"),
            lambda: __import__("kaggle_secrets").UserSecretsClient().get_secret("OPENAI_API_KEY")):
    if key: break
    try: key = get() or ""
    except Exception: pass
if not key:
    import getpass; key = getpass.getpass("OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = key.strip()
print(open("results/p26b/rule_gpt4o.md").read())

In [ ]:
for form in ("free", "json"):
    !python -u kit_tools/p26_openai.py --save {SAVE} --form {form} --tag gpt4o 2>&1 | tail -6

In [ ]:
import json
for form in ("free", "json"):
    !python -I kit_tools/p26_score.py results/p26b/sample.jsonl results/p26b/labels.json {SAVE}/score_{form}_gpt4o.json {SAVE}/p26b_gpt4o_{form}_replies.json > /dev/null
s = json.load(open(f"{SAVE}/score_json_gpt4o.json"))
acc = s["validator"]["accepted"]
right, wrong = acc["right"], acc["count"] - acc["right"]
print(f"Forced form (decides): found {s['found_of_labelled']} of 56; validator stored {acc['count']}, {right} right and {wrong} wrong")
print("Condition 1 (at least 3 right per wrong):", "met" if right >= 3 * wrong and acc["count"] else "not met")
print("Condition 2 (at least 19 found):", "met" if s["found_of_labelled"] >= 19 else "not met")
f = json.load(open(f"{SAVE}/score_free_gpt4o.json"))
print(f"Free form (for the record): found {f['found_of_labelled']} of 56; stored {f['validator']['accepted']['count']}, {f['validator']['accepted']['right']} right")

In [ ]:
os.chdir(WORK)
from IPython.display import FileLink, display
for f in ("p26b_gpt4o_free_replies.json", "p26b_gpt4o_json_replies.json", "score_free_gpt4o.json", "score_json_gpt4o.json"):
    display(FileLink(f"engram_results/{f}"))

Send Claude the four files above. Claude rescored them from the replies before reporting, so the result does not rest on this notebook's printout.